# Filter sample points by MOLCA 2024 coverage and re-extract class values

Reads the 2019 photo-interpreted reference points GeoJSON, keeps only those that fall inside a valid MOLCA 2024 tile and on a non-nodata pixel, and re-samples the 2024 class value at each kept point for comparison against the original `molca_class`.

**To re-run for another region**: change `REGION` and the input/output paths in Cell 1. The rest of the notebook is region-agnostic.

## Cell 1 — Region paths (only thing to edit between regions)

In [1]:
import os
from pyproj import datadir
os.environ["PROJ_LIB"] = datadir.get_data_dir()
from rasterio.crs import CRS
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.warp import transform as warp_transform

REGION = "Siberia"  # "Africa", "Amazon_Extension", "Siberia"

PROJECT_DIR = r"d:\06_Polimi\2025-2026\Land Cover\Validation\sample-design"
os.makedirs(rf"{PROJECT_DIR}\Samples_2024", exist_ok=True)

INPUT_BY_REGION = {
    "Africa": "africa_static_area_2019_photointerpreted_UniGE_merged.geojson",
    "Amazon_Extension": "amazonia_static_area_2019_photointerpreted_UniTN_merged.geojson",
    "Siberia": "siberia_static_area_2019_photointerpreted_UniPV_merged.geojson",
}

if REGION not in INPUT_BY_REGION:
    raise ValueError(f"Unsupported REGION '{REGION}'. Choose one of: {list(INPUT_BY_REGION)}")

input_name = INPUT_BY_REGION[REGION]
INPUT_GEOJSON = rf"{PROJECT_DIR}\Samples_2019\{input_name}"
MOLCA_GPKG = rf"{PROJECT_DIR}\MOLCA\{REGION}_2024_UTM\molcx_{REGION.lower()}_2024_s2_exist.gpkg"
MOLCA_TIFFS = rf"{PROJECT_DIR}\MOLCA\{REGION}_2024_UTM\Tiffs"
LEGEND_CSV = rf"{PROJECT_DIR}\config\MOLCA_legend.csv"

OUTPUT_PREFIX_BY_REGION = {
    "Africa": "africa",
    "Amazon_Extension": "amazonia",
    "Siberia": "siberia",
}

output_prefix = OUTPUT_PREFIX_BY_REGION[REGION]
OUTPUT_GEOJSON = rf"{PROJECT_DIR}\Samples_2024\{output_prefix}_static_area_2024_molca_filtered.geojson"

NODATA_CLASS = 255

print("Region:       ", REGION)
print("Input:        ", INPUT_GEOJSON)
print("MOLCA gpkg:   ", MOLCA_GPKG)
print("MOLCA tiffs:  ", MOLCA_TIFFS)
print("Legend:       ", LEGEND_CSV)
print("Output:       ", OUTPUT_GEOJSON)

Region:        Siberia
Input:         d:\06_Polimi\2025-2026\Land Cover\Validation\sample-design\Samples_2019\siberia_static_area_2019_photointerpreted_UniPV_merged.geojson
MOLCA gpkg:    d:\06_Polimi\2025-2026\Land Cover\Validation\sample-design\MOLCA\Siberia_2024_UTM\molcx_siberia_2024_s2_exist.gpkg
MOLCA tiffs:   d:\06_Polimi\2025-2026\Land Cover\Validation\sample-design\MOLCA\Siberia_2024_UTM\Tiffs
Legend:        d:\06_Polimi\2025-2026\Land Cover\Validation\sample-design\config\MOLCA_legend.csv
Output:        d:\06_Polimi\2025-2026\Land Cover\Validation\sample-design\Samples_2024\siberia_static_area_2024_molca_filtered.geojson


## Cell 2 — Load and preview inputs

In [6]:
points = gpd.read_file(INPUT_GEOJSON)
print("Points shape:", points.shape, " CRS:", points.crs)
points.head()

Points shape: (54989, 3)  CRS: EPSG:4326


,molca_class,molca_label,geometry
0,12,Bareland,POINT (66.68447 66.52036)
1,12,Bareland,POINT (66.7935 66.47648)
2,12,Bareland,POINT (66.79334 66.47648)
3,12,Bareland,POINT (66.61103 66.60306)
4,12,Bareland,POINT (66.81397 66.47437)


In [7]:
gdf = gpd.read_file(MOLCA_GPKG)
print("gpkg rows:", len(gdf), "  unique Tile_ID:", gdf['Tile_ID'].nunique(), "  CRS:", gdf.crs)
gdf.head()

gpkg rows: 1995   unique Tile_ID: 187   CRS: EPSG:4326


,Tile_ID,index_right,Existing_data_location,layer,tile_crs,tile_resolution,geometry
0,41VPH,118,/g100_work/pMI25_DICA/qxu00001/ESA_CCI_HRLC_Ph...,ESRI_Annual_LULC_2024,EPSG:32641,30,"POLYGON ((64.86801 61.3216, 66.9149 61.27798, ..."
1,41VPH,51,/g100_work/pMI25_DICA/qxu00001/ESA_CCI_HRLC_Ph...,WorldCover_2021,EPSG:32641,30,"POLYGON ((64.86801 61.3216, 66.9149 61.27798, ..."
2,41VPH,62,/g100_work/pMI25_DICA/qxu00001/ESA_CCI_HRLC_Ph...,WorldCover_2021,EPSG:32641,30,"POLYGON ((64.86801 61.3216, 66.9149 61.27798, ..."
3,41VPH,115,/g100_work/pMI25_DICA/qxu00001/ESA_CCI_HRLC_Ph...,ESRI_Annual_LULC_2024,EPSG:32641,30,"POLYGON ((64.86801 61.3216, 66.9149 61.27798, ..."
4,41VPH,97,/g100_work/pMI25_DICA/qxu00001/ESA_CCI_HRLC_Ph...,GLC_FCS30D_2022,EPSG:32641,30,"POLYGON ((64.86801 61.3216, 66.9149 61.27798, ..."


In [8]:
legend_df = pd.read_csv(LEGEND_CSV)
legend_df

,Class code,Label
0,5,Shrubland
1,7,Grassland
2,8,Cropland
3,13,Built-up
4,12,Bareland
5,16,Permanent ice and snow
6,15,Water
7,9,Wetland
8,11,Lichens and mosses
9,20,Forest


## Cell 3 — Peek at one tile to verify the legend

Confirm the unique class values found in the raster are a subset of the legend's `Class code` column. If unfamiliar codes appear, stop and investigate.

In [9]:
sample_tile = next((tile_id for tile_id in gdf['Tile_ID'].dropna().astype(str).unique() if os.path.exists(rf"{MOLCA_TIFFS}\MOLCA_{tile_id}_F.tif")), None)
if sample_tile is None:
    raise FileNotFoundError("No matching MOLCA GeoTIFF was found for the Tile_ID values loaded in gdf.")
print("Sample tile:    ", sample_tile)
with rasterio.open(rf"{MOLCA_TIFFS}\MOLCA_{sample_tile}_F.tif") as src:
    arr = src.read(1)
    print("CRS:           ", src.crs)
    print("Resolution:    ", src.res)
    print("Nodata declared:", src.nodata)
    print("Bounds:        ", src.bounds)
print("Unique values in this tile:", np.unique(arr))
print("\nLegend for reference:")
print(legend_df.to_string(index=False))

Sample tile:     41VPH
CRS:            EPSG:32641
Resolution:     (30.0, 30.0)
Nodata declared: 255.0
Bounds:         BoundingBox(left=600000.0, bottom=6690240.0, right=709800.0, top=6800040.0)
Unique values in this tile: [  7   8   9  11  12  13  15  20 255]

Legend for reference:
 Class code                  Label
          5              Shrubland
          7              Grassland
          8               Cropland
         13               Built-up
         12               Bareland
         16 Permanent ice and snow
         15                  Water
          9                Wetland
         11     Lichens and mosses
         20                 Forest
        255                No data


## Cell 4 — Collapse gpkg to unique tiles + verify .tif exists

The gpkg lists each tile once per source dataset (≈ 2,174 rows for ~252 unique tiles). Reduce to one polygon per `Tile_ID`, then check that each tile's `.tif` actually exists on disk.

In [10]:
tiles = gdf[['Tile_ID', 'geometry']].drop_duplicates('Tile_ID').reset_index(drop=True)
tiles['tif_path'] = tiles['Tile_ID'].apply(lambda t: rf"{MOLCA_TIFFS}\MOLCA_{t}_F.tif")
tiles['tif_exists'] = tiles['tif_path'].apply(os.path.exists)
print("Tiles in gpkg:", len(tiles), "  with .tif on disk:", int(tiles['tif_exists'].sum()))
missing = tiles.loc[~tiles['tif_exists'], 'Tile_ID'].tolist()
if missing:
    print("Tiles listed in gpkg but missing on disk:", missing)
tiles = tiles[tiles['tif_exists']].drop(columns='tif_exists').reset_index(drop=True)
tiles.head()

Tiles in gpkg: 187   with .tif on disk: 187


,Tile_ID,geometry,tif_path
0,41VPH,"POLYGON ((64.86801 61.3216, 66.9149 61.27798, ...",d:\06_Polimi\2025-2026\Land Cover\Validation\s...
1,41VPJ,"POLYGON ((64.92321 62.21844, 67.03031 62.17316...",d:\06_Polimi\2025-2026\Land Cover\Validation\s...
2,41VPK,"POLYGON ((64.98231 63.11567, 67.15384 63.06863...",d:\06_Polimi\2025-2026\Land Cover\Validation\s...
3,41VPL,"POLYGON ((65.04568 64.01275, 67.28624 63.96384...",d:\06_Polimi\2025-2026\Land Cover\Validation\s...
4,41WPM,"POLYGON ((65.11372 64.90915, 67.42838 64.85824...",d:\06_Polimi\2025-2026\Land Cover\Validation\s...


## Cell 5 — Spatial join: point → tile

Each surviving point gets the `Tile_ID` of the tile that contains it. Rows where `Tile_ID` is NaN are outside MOLCA 2024 coverage and will be dropped.

In [11]:
tiles = tiles.to_crs(points.crs)
joined = gpd.sjoin(points, tiles[['Tile_ID', 'geometry']], predicate="within", how="left")
# Drop duplicate matches from tile-edge overlaps (rare); keep first.
joined = joined[~joined.index.duplicated(keep='first')]
n_outside = int(joined['Tile_ID'].isna().sum())
print(f"Points outside MOLCA 2024 coverage: {n_outside} / {len(joined)}")
joined.head()

Points outside MOLCA 2024 coverage: 29723 / 54989


,molca_class,molca_label,geometry,index_right,Tile_ID
0,12,Bareland,POINT (66.68447 66.52036),6.0,41WPP
1,12,Bareland,POINT (66.7935 66.47648),6.0,41WPP
2,12,Bareland,POINT (66.79334 66.47648),6.0,41WPP
3,12,Bareland,POINT (66.61103 66.60306),6.0,41WPP
4,12,Bareland,POINT (66.81397 66.47437),6.0,41WPP


In [12]:
# Show a few of the points that fell outside coverage, for sanity
joined[joined['Tile_ID'].isna()].head()

,molca_class,molca_label,geometry,index_right,Tile_ID
100,12,Bareland,POINT (66.04412 53.82015),NaN,NaN
101,12,Bareland,POINT (66.19671 53.56288),NaN,NaN
102,12,Bareland,POINT (66.19167 53.56288),NaN,NaN
103,12,Bareland,POINT (66.04285 53.81889),NaN,NaN
104,12,Bareland,POINT (66.20176 53.60576),NaN,NaN


## Cell 6 — Raster sampling per tile

For each tile group: open the GeoTIFF once, reproject the group's lon/lat into the tile's UTM CRS, batch-sample pixel values.

In [13]:
joined['molca_class_2024'] = np.nan
inside = joined.dropna(subset=['Tile_ID'])
n_tiles = inside['Tile_ID'].nunique()
print(f"Sampling {len(inside)} points across {n_tiles} tiles...")

for tile_id, grp in inside.groupby('Tile_ID'):
    lons = grp.geometry.x.tolist()
    lats = grp.geometry.y.tolist()
    with rasterio.open(rf"{MOLCA_TIFFS}\MOLCA_{tile_id}_F.tif") as src:
        xs, ys = warp_transform("EPSG:4326", src.crs, lons, lats)
        vals = [v[0] for v in src.sample(list(zip(xs, ys)))]
    joined.loc[grp.index, 'molca_class_2024'] = vals

print("Done.  Unique sampled values:", sorted(joined['molca_class_2024'].dropna().unique().tolist()))

Sampling 25266 points across 119 tiles...
Done.  Unique sampled values: [5.0, 7.0, 9.0, 11.0, 12.0, 13.0, 15.0, 20.0, 255.0]


## Cell 7 — Filter nodata, attach label, compare, write GeoJSON

In [15]:
n_nodata = int(((joined['molca_class_2024'] == NODATA_CLASS)).sum())

kept = joined[joined['molca_class_2024'].notna() & (joined['molca_class_2024'] != NODATA_CLASS)].copy()
kept['molca_class_2024'] = kept['molca_class_2024'].astype(int)

label_map = dict(zip(legend_df['Class code'], legend_df['Label']))
kept['molca_label_2024'] = kept['molca_class_2024'].map(label_map)
kept['class_agrees'] = kept['molca_class'] == kept['molca_class_2024']

out_cols = ['molca_class', 'molca_label', 'molca_class_2024', 'molca_label_2024', 'class_agrees', 'Tile_ID', 'geometry']
out = kept[out_cols].copy()
out = gpd.GeoDataFrame(out, geometry='geometry', crs=points.crs)

os.makedirs(os.path.dirname(OUTPUT_GEOJSON), exist_ok=True)
if os.path.exists(OUTPUT_GEOJSON):
    os.remove(OUTPUT_GEOJSON)
out.to_file(OUTPUT_GEOJSON, driver="GeoJSON")
print("Wrote:", OUTPUT_GEOJSON)
out.head()

Wrote: d:\06_Polimi\2025-2026\Land Cover\Validation\sample-design\Samples_2024\siberia_static_area_2024_molca_filtered.geojson


,molca_class,molca_label,molca_class_2024,molca_label_2024,class_agrees,Tile_ID,geometry
21,12,Bareland,20,Forest,False,41WPP,POINT (67.67257 66.56488)
22,12,Bareland,20,Forest,False,41WPP,POINT (67.32711 66.50053)
25,12,Bareland,20,Forest,False,41WPP,POINT (67.62658 66.55383)
31,12,Bareland,13,Built-up,False,41WPP,POINT (66.68155 66.52897)
38,12,Bareland,13,Built-up,False,41WPP,POINT (66.6822 66.52279)


## Cell 8 — Summary

In [16]:
total = len(points)
kept_n = len(kept)
agree_n = int(kept['class_agrees'].sum())
agree_pct = 100.0 * agree_n / kept_n if kept_n else 0.0
assert n_outside + n_nodata + kept_n == total, "Counts do not add up"

print(f"Total input:                {total}")
print(f"Dropped (outside MOLCA):    {n_outside}")
print(f"Dropped (nodata, class 255):{n_nodata}")
print(f"Kept:                       {kept_n}")
print(f"Agreement with 2019 label:  {agree_n} / {kept_n}  ({agree_pct:.1f}%)")

Total input:                54989
Dropped (outside MOLCA):    29723
Dropped (nodata, class 255):15411
Kept:                       9855
Agreement with 2019 label:  7793 / 9855  (79.1%)


In [17]:
# Count points in each class (2024 classification)
kept_by_class = kept['molca_class_2024'].value_counts().sort_index()
print("Points in each class (2024):")
print(kept_by_class)
print(f"\nTotal: {kept_by_class.sum()}")

Points in each class (2024):
molca_class_2024
5       12
7      461
9     1537
11     449
12     444
13     504
15    1874
20    4574
Name: count, dtype: int64

Total: 9855


## Cell 9 — Per-class tile coverage vs. tiles hosting the 1,192 filtered points

Restricted to the 7 classes actually present in MOLCA 2024 (7, 8, 9, 12, 13, 15, 20). For each class:
- **tiles_with_class**: tiles (out of 252) whose MOLCA 2024 raster contains at least one valid pixel of that class
- **tiles_with_kept_pts**: of those tiles, how many host at least one of the 1,192 kept points whose 2024-resampled class equals this class

In [18]:
# Scan every tile once: record which classes its raster contains (excluding nodata)
tile_classes = {}
for _, row in tiles.iterrows():
    with rasterio.open(row['tif_path']) as src:
        u = np.unique(src.read(1))
    tile_classes[row['Tile_ID']] = set(int(v) for v in u if v != NODATA_CLASS)

# Tiles that host at least one of the 1,192 kept points, grouped by the point's 2024-resampled class
tiles_with_kept_by_class = (
    kept.groupby('molca_class_2024')['Tile_ID']
        .apply(lambda s: set(s.unique()))
        .to_dict()
)

# Only the 7 classes actually present in MOLCA 2024
classes_present_2024 = sorted(kept['molca_class_2024'].unique().tolist())
label_map = dict(zip(legend_df['Class code'], legend_df['Label']))

rows = []
for code in classes_present_2024:
    tiles_with_class = {t for t, classes in tile_classes.items() if code in classes}
    tiles_with_kept  = tiles_with_kept_by_class.get(code, set())
    rows.append({
        'class_code': code,
        'label': label_map.get(code, '?'),
        'tiles_with_class': len(tiles_with_class),
        'tiles_with_kept_pts': len(tiles_with_kept),
    })

summary = pd.DataFrame(rows)
print(f"Total tiles considered: {len(tiles)}")
print(f"Kept points:            {len(kept)}  across {kept['Tile_ID'].nunique()} tiles")
print(summary.to_string(index=False))

Total tiles considered: 187
Kept points:            9855  across 105 tiles
 class_code              label  tiles_with_class  tiles_with_kept_pts
          5          Shrubland                 4                    2
          7          Grassland               182                   47
          9            Wetland               186                   58
         11 Lichens and mosses               171                   24
         12           Bareland               184                   35
         13           Built-up               147                   14
         15              Water               187                   74
         20             Forest               163                   70


## Cell 10 — Distribution of kept points per tile, by class

For each 2024 class, bucket tiles by how many kept points they host: 1–2, 3–5, 6–9, 10+. Also report the max points in a single tile.

In [19]:
# Points-per-tile counts for each 2024 class, then bucket
bins   = [0, 2, 5, 9, np.inf]
labels = ['1-2', '3-5', '6-9', '10+']

rows = []
for code in sorted(kept['molca_class_2024'].unique().tolist()):
    sub = kept[kept['molca_class_2024'] == code]
    per_tile = sub.groupby('Tile_ID').size()  # points per tile (only tiles with >=1 point)
    buckets  = pd.cut(per_tile, bins=bins, labels=labels).value_counts().reindex(labels, fill_value=0)
    rows.append({
        'class_code': code,
        'label':      label_map.get(code, '?'),
        'points':     int(per_tile.sum()),
        'tiles':      int(per_tile.size),
        '1-2':        int(buckets['1-2']),
        '3-5':        int(buckets['3-5']),
        '6-9':        int(buckets['6-9']),
        '10+':        int(buckets['10+']),
        'max_per_tile': int(per_tile.max()),
        'median_per_tile': float(per_tile.median()),
    })

dist = pd.DataFrame(rows)
print(dist.to_string(index=False))

 class_code              label  points  tiles  1-2  3-5  6-9  10+  max_per_tile  median_per_tile
          5          Shrubland      12      2    1    0    0    1            11              6.0
          7          Grassland     461     47   17   14    4   12            92              4.0
          9            Wetland    1537     58   15    6    5   32           189             11.5
         11 Lichens and mosses     449     24    5    5    1   13            95             11.5
         12           Bareland     444     35   11    6    5   13            61              6.0
         13           Built-up     504     14    2    1    3    8           120             17.5
         15              Water    1874     74   17   15    8   34            96              9.0
         20             Forest    4574     70    7    2    6   55           320             30.5
